In [1]:
import sys
import subprocess

def check_lib(package_name):
    try:
        dist = __import__(package_name)
        print(f"✅ {package_name} is installed. Version: {getattr(dist, '__version__', 'Unknown')}")
        return True
    except ImportError:
        print(f"❌ {package_name} is MISSING.")
        return False

print("--- Python Library Check ---")
opencv_ok = check_lib("cv2")
numpy_ok = check_lib("numpy")

if opencv_ok:
    import cv2
    # Check specifically for ArUco module in OpenCV
    if hasattr(cv2, 'aruco'):
        print("✅ ArUco module found in cv2.")
    else:
        print("❌ cv2 found, but ArUco module is MISSING (Need opencv-contrib-python).")

print("\n--- System Hardware Support Check ---")
# Check for libatlas (needed for fast math on ARM)
try:
    lib_check = subprocess.check_output("ldconfig -p | grep libatlas", shell=True)
    print("✅ libatlas-base-dev is linked and ready.")
except subprocess.CalledProcessError:
    print("❌ libatlas-base-dev is MISSING (Run 'sudo apt install libatlas-base-dev').")

--- Python Library Check ---
✅ cv2 is installed. Version: 4.5.4
✅ numpy is installed. Version: 1.21.5
✅ ArUco module found in cv2.

--- System Hardware Support Check ---
✅ libatlas-base-dev is linked and ready.


In [ ]:
import sys
import subprocess
from IPython.display import display, Javascript
import math
import os
import threading
import time
from flask import Flask, Response
import cv2
import cv2.aruco as aruco
import numpy as np

# 1. Setup Environment and Legacy ArUco Parameters
os.environ["OPENCV_LOG_LEVEL"] = "ERROR"
app = Flask(__name__)

# LEGACY initialization for OpenCV 4.5.4 (Fixes image_b98640)
aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)
parameters = aruco.DetectorParameters_create()

# 2. Initialize Camera with V4L2 Backend (Fixes image_b985fc)
video_in = cv2.VideoCapture(0, cv2.CAP_V4L2)
# --- RESOLUTION SETTINGS ---
# 640x480 forces a 4:3 ratio. If the image still looks squished after running this, 
# change these to a 16:9 ratio like 640x360 or 1280x720 to match your webcam's native sensor.
video_in.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
video_in.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# Define our dashboard colors and formal font
LIGHT_BLUE = (230, 216, 173) 
GREEN = (0, 255, 0)
FONT = cv2.FONT_HERSHEY_DUPLEX

def gen_frames():
    while True:
        success, frame = video_in.read()
        if not success:
            break
        else:
            frame = cv2.flip(frame, -1)
            # Dynamically grab the ACTUAL size the camera is outputting
            frame_h, frame_w = frame.shape[:2]
            
            # --- CREATE THE DYNAMIC DASHBOARD CANVAS ---
            # Make the canvas exactly 120 pixels taller than the camera frame
            canvas = np.zeros((frame_h + 120, frame_w, 3), dtype=np.uint8)
            
            # Paste the exact, unscaled camera frame into the top
            canvas[0:frame_h, 0:frame_w] = frame
            
            # Calculate the absolute center of the frame dynamically
            frame_center_x, frame_center_y = int(frame_w / 2), int(frame_h / 2)
            
            # --- DRAW FRAME CENTER (LIGHT BLUE) ---
            cv2.drawMarker(canvas, (frame_center_x, frame_center_y), LIGHT_BLUE, 
                           markerType=cv2.MARKER_CROSS, markerSize=10, thickness=1)
            cv2.circle(canvas, (frame_center_x, frame_center_y), 2, LIGHT_BLUE, -1)
            
            # Define exact Y-coordinates for the text rows so they stay in the black bar
            row1_y = frame_h + 40
            row2_y = frame_h + 80
            
            # --- WRITE FRAME STATS (LIGHT BLUE) ---
            cv2.putText(canvas, "Frame", (10, row1_y), FONT, 0.45, LIGHT_BLUE, 1)
            cv2.putText(canvas, f"C: {frame_center_x},{frame_center_y}", (110, row1_y), FONT, 0.45, LIGHT_BLUE, 1)
            cv2.putText(canvas, f"W: {frame_w} H: {frame_h}", (250, row1_y), FONT, 0.45, LIGHT_BLUE, 1)
            cv2.putText(canvas, f"Area: {frame_w * frame_h} px^2", (410, row1_y), FONT, 0.45, LIGHT_BLUE, 1)

            # --- ARUCO DETECTION LOGIC ---
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            corners, ids, rejected = aruco.detectMarkers(gray, aruco_dict, parameters=parameters)

            if ids is not None:
                for i in range(len(ids)):
                    c = corners[i][0]
                    
                    # Draw clean polygon box
                    pts = c.astype(np.int32)
                    cv2.polylines(canvas, [pts], isClosed=True, color=GREEN, thickness=2)
                    
                    top_left, top_right, bottom_right, bottom_left = c[0], c[1], c[2], c[3]
                    
                    # Math Calculations
                    width = math.hypot(top_right[0] - top_left[0], top_right[1] - top_left[1])
                    height = math.hypot(bottom_left[0] - top_left[0], bottom_left[1] - top_left[1])
                    area = width * height
                    center_x, center_y = int(c[:, 0].mean()), int(c[:, 1].mean())
                    
                    # --- DRAW ARUCO CENTER (GREEN) ---
                    cv2.drawMarker(canvas, (center_x, center_y), GREEN, 
                                   markerType=cv2.MARKER_CROSS, markerSize=15, thickness=2)
                    cv2.circle(canvas, (center_x, center_y), 3, GREEN, -1)
                    
                    # --- WRITE ARUCO STATS (GREEN) ---
                    cv2.putText(canvas, f"ID: {ids[i][0]}", (10, row2_y), FONT, 0.45, GREEN, 1)
                    cv2.putText(canvas, f"C: {center_x},{center_y}", (110, row2_y), FONT, 0.45, GREEN, 1)
                    cv2.putText(canvas, f"W: {int(width)} H: {int(height)}", (250, row2_y), FONT, 0.45, GREEN, 1)
                    cv2.putText(canvas, f"Area: {int(area)} px^2", (410, row2_y), FONT, 0.45, GREEN, 1)
                    
            else:
                cv2.putText(canvas, "Searching for Markers...", (10, row2_y), 
                            FONT, 0.45, (0, 0, 255), 1)

            # --- Prepare the CANVAS for Flask ---
            ret, buffer = cv2.imencode('.jpg', canvas)
            frame_bytes = buffer.tobytes()
            yield (b'--frame\r\n'
                   b'Content-Type: image/jpeg\r\n\r\n' + frame_bytes + b'\r\n')

@app.route('/video_feed')
def video_feed():
    return Response(gen_frames(), mimetype='multipart/x-mixed-replace; boundary=frame')

@app.route('/')
def index():
    return '''
    <html>
        <head>
            <style>
                body { margin: 0; background: #000; display: flex; justify-content: center; align-items: center; height: 100vh; }
                img { width: 640px; height: 600; border: 2px solid #00ff00; image-rendering: pixelated; }
            </style>
        </head>
        <body><img src="/video_feed"></body>
    </html>
    '''

def launch_and_run():
    # Start Flask in background
    threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000, debug=False, threaded=True)).start()
    time.sleep(2)
    # Open the window on YOUR laptop browser (ensure IP matches your board)
    display(Javascript('window.open("http://192.168.2.99:5000", "_blank", "width=660,height=610");'))
    print("ArUco Server Online")

if __name__ == '__main__':
    launch_and_run()
    
    try:
        while True:
            time.sleep(0.1)
    except KeyboardInterrupt:
        print("\nCell interrupted. Stopping logs.")

 * Serving Flask app '__main__' (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://127.0.0.1:5000 (Press CTRL+C to quit)


<IPython.core.display.Javascript object>

192.168.2.1 - - [27/Mar/2026 20:02:31] "GET / HTTP/1.1" 200 -


ArUco Server Online


192.168.2.1 - - [27/Mar/2026 20:02:33] "GET /video_feed HTTP/1.1" 200 -
192.168.2.1 - - [27/Mar/2026 20:02:34] "GET /favicon.ico HTTP/1.1" 404 -
